# Notebook 7: Initialization — Exploding & Vanishing Gradients

---

## Learning Objectives

By the end of this notebook you will be able to:

- Explain why weight initialization matters and why all-zeros fails
- Derive and apply Xavier/Glorot initialization
- Derive and apply He/Kaiming initialization for ReLU networks
- Identify vanishing and exploding gradients from diagnostic plots
- Apply gradient clipping to prevent exploding gradients
- Visualize activation and gradient distributions across layers

## Prerequisites

- Notebook 03 (Backpropagation)
- Notebook 05 (Optimization)
- Understanding of matrix multiplication and variance

## Table of Contents

1. [Why Initialization Matters](#1-why-initialization-matters)
2. [Random Initialization Pitfalls](#2-random-initialization-pitfalls)
3. [Xavier/Glorot Initialization](#3-xavierglorot-initialization)
4. [He/Kaiming Initialization](#4-hekaiming-initialization)
5. [Vanishing Gradients](#5-vanishing-gradients)
6. [Exploding Gradients & Gradient Clipping](#6-exploding-gradients--gradient-clipping)
7. [Code: Activation Distributions with Different Inits](#7-code-activation-distributions)
8. [Code: Gradient Magnitudes with Different Inits](#8-code-gradient-magnitudes)
9. [Code: Gradient Clipping Demo](#9-code-gradient-clipping-demo)
10. [Exercise: Experiment with Initializations](#10-exercise)
11. [Common Mistakes & Debugging Tips](#11-common-mistakes--debugging-tips)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Why Initialization Matters

### The All-Zeros Problem (Symmetry Breaking)

If all weights are initialized to the same value (e.g., zero), every neuron in a layer computes the **exact same output**. During backpropagation, they all receive the **exact same gradient**. As a result, they remain identical forever — the network effectively has only one neuron per layer.

$$\text{If } w_1 = w_2 = \dots = w_n, \text{ then } \nabla_{w_1} L = \nabla_{w_2} L = \dots = \nabla_{w_n} L$$

**This is called the symmetry problem.** Random initialization breaks this symmetry.

### What Can Go Wrong

- **Too small** initialization: Activations shrink to zero layer by layer (vanishing).
- **Too large** initialization: Activations explode layer by layer (exploding/saturating).
- **Just right**: Activations maintain a healthy variance throughout the network.

In [ ]:
# Demonstrate the symmetry problem with zero initialization
torch.manual_seed(42)

# Model with zero init
model_zero = nn.Sequential(
    nn.Linear(5, 4), nn.ReLU(),
    nn.Linear(4, 4), nn.ReLU(),
    nn.Linear(4, 1)
)

# Set all weights to zero
for p in model_zero.parameters():
    nn.init.zeros_(p)

x = torch.randn(3, 5)
y = torch.randn(3, 1)

# Forward pass
out = model_zero(x)
loss = nn.MSELoss()(out, y)
loss.backward()

# Check: all neurons in a layer have identical gradients
layer1_weight_grad = model_zero[0].weight.grad
print("Layer 1 weight gradients (all rows identical):")
print(layer1_weight_grad)
print(f"\nAre all rows the same? {torch.allclose(layer1_weight_grad[0], layer1_weight_grad[1])}")

---
## 2. Random Initialization Pitfalls

Simply using random values (e.g., `N(0, 1)`) does not solve the problem. The **scale** of initialization matters critically.

Consider a layer $z = Wx$ where $W \in \mathbb{R}^{m \times n}$, $x \in \mathbb{R}^n$:

$$\text{Var}(z_j) = n \cdot \text{Var}(w) \cdot \text{Var}(x)$$

(assuming $w$ and $x$ are independent with zero mean).

- If $\text{Var}(w) = 1$ and $n = 1000$: the output variance is $1000 \times$ the input variance.
- Over $L$ layers, activations scale as $(n \cdot \text{Var}(w))^L$ — exponential blowup or shrinkage.

In [ ]:
# Show how activations explode with N(0, 1) init in a deep network
torch.manual_seed(42)

n_layers = 20
hidden_size = 256
x = torch.randn(100, hidden_size)

activation_stds_large = [x.std().item()]
activation_stds_small = [x.std().item()]

h_large = x.clone()
h_small = x.clone()

for i in range(n_layers):
    # Too large: N(0, 1)
    W_large = torch.randn(hidden_size, hidden_size) * 1.0
    h_large = torch.tanh(h_large @ W_large)
    activation_stds_large.append(h_large.std().item())
    
    # Too small: N(0, 0.01)
    W_small = torch.randn(hidden_size, hidden_size) * 0.01
    h_small = torch.tanh(h_small @ W_small)
    activation_stds_small.append(h_small.std().item())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(range(len(activation_stds_large)), activation_stds_large, color='red', alpha=0.7)
axes[0].set_title('N(0, 1) Init: Activations Saturate', fontsize=12)
axes[0].set_xlabel('Layer')
axes[0].set_ylabel('Std of Activations')
axes[0].grid(True, alpha=0.3)

axes[1].bar(range(len(activation_stds_small)), activation_stds_small, color='blue', alpha=0.7)
axes[1].set_title('N(0, 0.01) Init: Activations Vanish', fontsize=12)
axes[1].set_xlabel('Layer')
axes[1].set_ylabel('Std of Activations')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Xavier/Glorot Initialization

Xavier initialization (Glorot & Bengio, 2010) is designed so that the variance of activations and gradients stays approximately constant across layers.

### Formula

$$\text{Var}(w) = \frac{2}{n_{\text{in}} + n_{\text{out}}}$$

where $n_{\text{in}}$ is the number of input units and $n_{\text{out}}$ is the number of output units of the layer.

Equivalently, sample from:

$$w \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}} + n_{\text{out}}}\right) \quad \text{or} \quad w \sim U\left(-\sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}, \sqrt{\frac{6}{n_{\text{in}} + n_{\text{out}}}}\right)$$

**Best for:** tanh and sigmoid activations (symmetric, approximately linear near zero).

**In PyTorch:**
```python
nn.init.xavier_uniform_(layer.weight)
nn.init.xavier_normal_(layer.weight)
```

---
## 4. He/Kaiming Initialization

He initialization (He et al., 2015) accounts for the fact that ReLU sets half of its inputs to zero, halving the variance.

### Formula

$$\text{Var}(w) = \frac{2}{n_{\text{in}}}$$

Sample from:

$$w \sim \mathcal{N}\left(0, \frac{2}{n_{\text{in}}}\right)$$

**Why the factor of 2?** ReLU kills ~50% of activations (sets them to 0), effectively halving the variance. The factor of 2 compensates for this.

**Best for:** ReLU and its variants (Leaky ReLU, ELU, etc.).

**In PyTorch:**
```python
nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
```

**Note:** PyTorch uses Kaiming (He) initialization by default for `nn.Linear` layers.

---
## 5. Vanishing Gradients

The vanishing gradient problem occurs when gradients become exponentially small as they flow backward through many layers.

### Why sigmoid/tanh suffer

For sigmoid: $\sigma'(x) = \sigma(x)(1 - \sigma(x)) \leq 0.25$

For tanh: $\text{tanh}'(x) = 1 - \text{tanh}^2(x) \leq 1$

Over $L$ layers, the gradient is multiplied by the derivative at each layer. With sigmoid:

$$\frac{\partial L}{\partial w_1} \propto \prod_{l=1}^{L} \sigma'(z_l) \leq 0.25^L$$

For $L = 10$: $0.25^{10} \approx 10^{-6}$ — the gradient is essentially zero.

**Solutions:**
- Use ReLU (gradient is either 0 or 1, no shrinkage).
- Use proper initialization (He init for ReLU).
- Use residual connections (skip connections).
- Use batch normalization.

---
## 6. Exploding Gradients & Gradient Clipping

The dual problem: gradients become exponentially large, causing:
- Huge parameter updates.
- NaN values in weights.
- Training instability.

This is especially common in recurrent networks (RNNs/LSTMs) where gradients flow through many time steps.

### Gradient Clipping

**Clip by norm:** Scale the gradient vector so its norm does not exceed a maximum.

$$\hat{g} = \begin{cases} g & \text{if } \|g\| \leq \theta \\ \theta \cdot \frac{g}{\|g\|} & \text{if } \|g\| > \theta \end{cases}$$

**In PyTorch:**
```python
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
```

**Clip by value:** Clamp each gradient component independently.
```python
torch.nn.utils.clip_grad_value_(model.parameters(), clip_value=0.5)
```

---
<a id='7-code-activation-distributions'></a>
## 7. Code: Activation Distributions with Different Inits

We build a deep network and visualize the distribution of activations at each layer for different initialization schemes.

In [ ]:
class DeepNetWithHooks(nn.Module):
    """Deep network that stores intermediate activations via hooks."""
    
    def __init__(self, n_layers=8, hidden_size=256, activation='relu'):
        super().__init__()
        layers = []
        for i in range(n_layers):
            in_dim = hidden_size
            layers.append(nn.Linear(in_dim, hidden_size))
            if activation == 'relu':
                layers.append(nn.ReLU())
            elif activation == 'sigmoid':
                layers.append(nn.Sigmoid())
            elif activation == 'tanh':
                layers.append(nn.Tanh())
        layers.append(nn.Linear(hidden_size, 1))
        self.net = nn.Sequential(*layers)
        self.activations = []
    
    def forward(self, x):
        self.activations = []
        for layer in self.net:
            x = layer(x)
            if isinstance(layer, (nn.ReLU, nn.Sigmoid, nn.Tanh)):
                self.activations.append(x.detach().clone())
        return x

def apply_init(model, init_type):
    """Apply a specific initialization to all linear layers."""
    for m in model.modules():
        if isinstance(m, nn.Linear):
            if init_type == 'small_random':
                nn.init.normal_(m.weight, std=0.01)
            elif init_type == 'large_random':
                nn.init.normal_(m.weight, std=1.0)
            elif init_type == 'xavier':
                nn.init.xavier_normal_(m.weight)
            elif init_type == 'kaiming':
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)

In [ ]:
torch.manual_seed(42)
x_test = torch.randn(500, 256)

init_types = ['small_random', 'large_random', 'xavier', 'kaiming']
init_labels = ['N(0, 0.01)', 'N(0, 1)', 'Xavier', 'He/Kaiming']

fig, axes = plt.subplots(len(init_types), 8, figsize=(20, 10))

for row, (init_type, label) in enumerate(zip(init_types, init_labels)):
    torch.manual_seed(42)
    model = DeepNetWithHooks(n_layers=8, hidden_size=256, activation='relu')
    apply_init(model, init_type)
    
    with torch.no_grad():
        _ = model(x_test)
    
    for col, act in enumerate(model.activations):
        ax = axes[row, col]
        values = act.numpy().flatten()
        # Clip for display if exploding
        values = np.clip(values, -10, 10)
        ax.hist(values, bins=50, density=True, alpha=0.7, color='steelblue')
        ax.set_xlim(-2, 2)
        ax.set_ylim(0, 10)
        if row == 0:
            ax.set_title(f'Layer {col+1}', fontsize=9)
        if col == 0:
            ax.set_ylabel(label, fontsize=10)
        ax.tick_params(labelsize=7)

plt.suptitle('Activation Distributions Across Layers (ReLU) with Different Initializations',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Observations:**
- `N(0, 0.01)`: Activations quickly collapse to zero (vanishing).
- `N(0, 1)`: Activations saturate at extreme values (exploding before ReLU clips to 0).
- Xavier: Good for tanh/sigmoid but slightly too small for ReLU.
- He/Kaiming: Maintains a healthy activation distribution throughout all layers with ReLU.

---
<a id='8-code-gradient-magnitudes'></a>
## 8. Code: Gradient Magnitudes with Different Inits

We measure the gradient norm at each layer to see how gradients flow backward.

In [ ]:
def compute_gradient_norms(init_type, activation='relu', n_layers=10, hidden_size=128):
    """Build a deep network, do a forward+backward pass, and return gradient norms per layer."""
    torch.manual_seed(42)
    model = DeepNetWithHooks(n_layers=n_layers, hidden_size=hidden_size, activation=activation)
    apply_init(model, init_type)
    
    x = torch.randn(64, hidden_size)
    y = torch.randn(64, 1)
    
    out = model(x)
    loss = nn.MSELoss()(out, y)
    loss.backward()
    
    grad_norms = []
    for name, param in model.named_parameters():
        if 'weight' in name and param.grad is not None:
            grad_norms.append(param.grad.norm().item())
    
    return grad_norms

# Compare gradient norms for different inits with ReLU
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ReLU networks
for init_type, label, color in [
    ('small_random', 'N(0, 0.01)', 'blue'),
    ('large_random', 'N(0, 1)', 'red'),
    ('xavier', 'Xavier', 'green'),
    ('kaiming', 'He/Kaiming', 'purple'),
]:
    norms = compute_gradient_norms(init_type, activation='relu')
    axes[0].plot(norms, 'o-', label=label, color=color, alpha=0.8)

axes[0].set_xlabel('Layer index (from input to output)')
axes[0].set_ylabel('Gradient Norm')
axes[0].set_title('Gradient Norms per Layer (ReLU)')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Sigmoid networks (to show vanishing gradients)
for init_type, label, color in [
    ('small_random', 'N(0, 0.01)', 'blue'),
    ('xavier', 'Xavier', 'green'),
    ('kaiming', 'He/Kaiming', 'purple'),
]:
    norms = compute_gradient_norms(init_type, activation='sigmoid')
    axes[1].plot(norms, 'o-', label=label, color=color, alpha=0.8)

axes[1].set_xlabel('Layer index (from input to output)')
axes[1].set_ylabel('Gradient Norm')
axes[1].set_title('Gradient Norms per Layer (Sigmoid) - Vanishing!')
axes[1].legend()
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

**Key takeaway:** With sigmoid, gradients vanish exponentially regardless of initialization. With ReLU and He initialization, gradients maintain a more consistent magnitude across layers.

---
## 9. Code: Gradient Clipping Demo

We simulate a scenario where gradients explode and show how gradient clipping stabilizes training.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# Create a dataset
X = torch.randn(200, 10)
y = (X[:, 0] * 2 + X[:, 1] - X[:, 2] + 0.3 * torch.randn(200))
loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)

def build_deep_model():
    """Build a deep model prone to gradient explosion."""
    torch.manual_seed(42)
    model = nn.Sequential(
        nn.Linear(10, 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 256), nn.ReLU(),
        nn.Linear(256, 1)
    )
    # Deliberately use large init to provoke gradient issues
    for m in model.modules():
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.5)
    return model

def train_with_clipping(clip_norm=None, epochs=50):
    """Train with optional gradient clipping. Return losses and grad norms."""
    model = build_deep_model()
    optimizer = optim.SGD(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    
    losses, grad_norms = [], []
    
    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_grad_norm = 0.0
        n = 0
        for xb, yb in loader:
            pred = model(xb).squeeze()
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            
            # Compute gradient norm BEFORE clipping
            total_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.data.norm(2).item() ** 2
            total_norm = total_norm ** 0.5
            epoch_grad_norm += total_norm
            
            # Apply gradient clipping
            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_norm)
            
            optimizer.step()
            epoch_loss += loss.item()
            n += 1
        
        losses.append(epoch_loss / n)
        grad_norms.append(epoch_grad_norm / n)
    
    return losses, grad_norms

losses_no_clip, gnorms_no_clip = train_with_clipping(clip_norm=None, epochs=80)
losses_clip_1, gnorms_clip_1 = train_with_clipping(clip_norm=1.0, epochs=80)
losses_clip_5, gnorms_clip_5 = train_with_clipping(clip_norm=5.0, epochs=80)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(losses_no_clip, label='No clipping', alpha=0.8)
axes[0].plot(losses_clip_5, label='Clip norm=5.0', alpha=0.8)
axes[0].plot(losses_clip_1, label='Clip norm=1.0', alpha=0.8)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss with Gradient Clipping')
axes[0].legend()
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].plot(gnorms_no_clip, label='No clipping', alpha=0.8)
axes[1].plot(gnorms_clip_5, label='Clip norm=5.0', alpha=0.8)
axes[1].plot(gnorms_clip_1, label='Clip norm=1.0', alpha=0.8)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Gradient Norm (before clipping)')
axes[1].set_title('Gradient Norms During Training')
axes[1].legend()
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
<a id='10-exercise'></a>
## 10. Exercise: Experiment with Initializations

**Task:**
1. Build a 6-layer ReLU MLP (input=20, hidden=128, output=1).
2. Train it with three initialization schemes: N(0, 0.01), Xavier, and He/Kaiming.
3. Plot the training loss curves.
4. Which initialization converges fastest? Which fails to train?

In [ ]:
# ============================================================
# EXERCISE: Complete the code below
# ============================================================

# torch.manual_seed(42)
# X_ex = torch.randn(300, 20)
# y_ex = X_ex[:, 0] + 2 * X_ex[:, 1] - X_ex[:, 2] + 0.2 * torch.randn(300)
# loader_ex = DataLoader(TensorDataset(X_ex, y_ex), batch_size=32, shuffle=True)
#
# def build_6layer_mlp():
#     # TODO: Build a 6-layer ReLU MLP (20 -> 128 -> 128 -> 128 -> 128 -> 128 -> 1)
#     pass
#
# init_schemes = ['small_random', 'xavier', 'kaiming']
# all_losses = {}
#
# for init in init_schemes:
#     torch.manual_seed(42)
#     model = build_6layer_mlp()
#     apply_init(model, init)
#     # TODO: Train for 200 epochs with Adam(lr=0.001)
#     # TODO: Store losses in all_losses[init]
#     pass
#
# # TODO: Plot all loss curves on one figure

### Solution

In [ ]:
torch.manual_seed(42)
X_ex = torch.randn(300, 20)
y_ex = X_ex[:, 0] + 2 * X_ex[:, 1] - X_ex[:, 2] + 0.2 * torch.randn(300)
loader_ex = DataLoader(TensorDataset(X_ex, y_ex), batch_size=32, shuffle=True)

def build_6layer_mlp():
    return nn.Sequential(
        nn.Linear(20, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 128), nn.ReLU(),
        nn.Linear(128, 1)
    )

init_schemes = ['small_random', 'xavier', 'kaiming']
init_labels_ex = ['N(0, 0.01)', 'Xavier', 'He/Kaiming']
all_losses = {}

for init, label in zip(init_schemes, init_labels_ex):
    torch.manual_seed(42)
    model = build_6layer_mlp()
    apply_init(model, init)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.MSELoss()
    losses = []
    
    for epoch in range(200):
        epoch_loss, n = 0.0, 0
        for xb, yb in loader_ex:
            pred = model(xb).squeeze()
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n += 1
        losses.append(epoch_loss / n)
    
    all_losses[label] = losses
    print(f"{label:15s} -> final loss: {losses[-1]:.4f}")

plt.figure(figsize=(10, 5))
for label, losses in all_losses.items():
    plt.plot(losses, label=label, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Effect of Initialization on Training (6-layer ReLU MLP)')
plt.legend()
plt.yscale('log')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 11. Common Mistakes & Debugging Tips

| Mistake | Symptom | Fix |
|---|---|---|
| Zero initialization | All neurons learn the same thing; model barely improves | Use Xavier or He initialization |
| Too large init (e.g., N(0,1)) | Activations saturate; loss stuck at high value | Use He init for ReLU, Xavier for tanh/sigmoid |
| Too small init (e.g., N(0,0.001)) | Activations collapse to 0; gradients vanish | Increase init scale; use proper init scheme |
| Using Xavier init with ReLU | Activations gradually shrink in deep networks | Switch to He/Kaiming init |
| Using sigmoid in deep networks | Vanishing gradients after a few layers | Switch to ReLU; use residual connections |
| Ignoring gradient norms | Silently exploding/vanishing gradients go undetected | Log gradient norms periodically |
| Not applying gradient clipping in RNNs | Training diverges randomly | Always clip gradients in RNN/LSTM training |

**Debugging checklist:**
1. Print activation statistics (mean, std) at each layer for the first batch.
2. Print gradient norms per layer — they should be roughly the same order of magnitude.
3. If loss is NaN: check for exploding gradients, reduce lr, add gradient clipping.
4. If loss is stuck: check for vanishing gradients, use He init + ReLU, add batch norm.
5. As a rule of thumb: **He init + ReLU + BatchNorm** is a safe default for most feedforward networks.